# Vocabulary Mapping Benchmarking with Real SNOMED CT RF2 Data

This notebook demonstrates how to benchmark SNOMED CT vocabulary mapping methods using real RF2 reference set data.

## Overview

Vocabulary mapping evaluates how well a method can convert SNOMED CT concepts to codes in other terminologies (ICD-10, LOINC, etc.).

**Key Changes from Previous Versions:**
- Uses *real* SNOMED CT mappings extracted from RF2 Simple Map reference set files
- Ground truth is actual mappings from the reference sets, not synthetic data
- Multiple mapper implementations with *different algorithms* for fair comparison

### Key Metrics:
- **Precision@K**: Fraction of top-K mapped codes that are correct
- **Recall@K**: Fraction of expected mappings found in top-K
- **Coverage Rate**: Fraction of all expected mappings successfully found
- **MRR (Mean Reciprocal Rank)**: Average rank of first correct mapping
- **F1@K**: Harmonic mean of precision and recall at K

In [ ]:
# Import required modules
import os

from snomed_methods.benchmarking.mock_methods import (
    approximate_mapper,
    random_mapper,
    simple_mapper,
)
from snomed_methods.benchmarking.vocabulary import (
    evaluate_mapper,
    generate_mapping_dataset,
)
from snomed_methods.benchmarking.vocabulary.dataset import load_simple_map_mappings

## Load Real SNOMED CT Data and Mappings from RF2

We extract actual mappings from the RF2 Simple Map reference set files.

In [ ]:
# Path to SNOMED CT RF2 data
rf2_path = os.environ.get(
    "SNOMED_RF2_PATH",
    "/workspaces/snomed_methods/uk_sct2cl_42.2.0/SnomedCT_InternationalRF2_PRODUCTION_20260201T120000Z",
)

print(f"RF2 Path: {rf2_path}")
print(f"Path exists: {os.path.exists(rf2_path)}")

In [ ]:
# Load actual mappings from RF2 Simple Map files
print("Loading SNOMED CT to target code mappings from RF2...")
real_mappings = load_simple_map_mappings(rf2_path)

num_mapped_concepts = sum(1 for targets in real_mappings.values() if len(targets) > 0)
total_concepts = len(real_mappings)
print(f"Total concepts with mappings: {num_mapped_concepts}/{total_concepts}")
print(f"Sample mappings ({min(5, len(real_mappings))} concepts):")

count = 0
for cui, targets in list(real_mappings.items())[:10]:
    if count >= 5:
        break
    print(f"  {cui}: {targets}")
    count += 1

In [ ]:
# Generate dataset using real RF2 mappings
dataset = generate_mapping_dataset(num_samples=50, snomed_dir=rf2_path)

print(f"Dataset size: {len(dataset)}")
print(f" Concepts with mappings: {sum(1 for s in dataset if s['has_mapping'])}")
print(f" Concepts without mappings: {sum(1 for s in dataset if not s['has_mapping'])}")

In [ ]:
# View first sample
sample = dataset[0]
print("First sample:")
print(f"  SNOMED CUI: {sample['snomed_cui']}")
print(f"  Has mapping: {sample['has_mapping']}")
print(f"  Target codes ({len(sample['target_codes'])}):")
for code in sample["target_codes"]:
    print(f"    - {code}")

## Create Multiple Mapper Functions with DIFFERENT Algorithms

Each mapper uses a different approach to find mappings:
- **simple_mapper**: Direct lookup (same algorithm as ground truth extraction)
- **approximate_mapper**: String similarity and partial matching (DIFFERENT from ground truth)
- **random_mapper**: Random sampling (baseline that should fail)

In [ ]:
# Get list of all concept IDs for approximate mapper
all_concept_ids = [s["snomed_cui"] for s in dataset]

# Define different mapping methods with DIFFERENT algorithms


def simple_mapper_func(snomed_cui: str) -> list:
    """Direct lookup mapper - same algorithm as ground truth extraction."""
    return simple_mapper(snomed_cui, real_mappings)


def approximate_mapper_func(snomed_cui: str) -> list:
    """Approximate/string-based matching mapper - DIFFERENT algorithm."""
    return approximate_mapper(snomed_cui, real_mappings, all_concept_ids)


def random_sampler_func(snomed_cui: str) -> list:
    """Random sampling mapper - baseline that should perform poorly."""
    return random_mapper(snomed_cui, real_mappings)

In [ ]:
# Test each mapper on the first concept
test_cui = dataset[0]["snomed_cui"]

print(f"Testing on SNOMED CUI: {test_cui}")
print(f"Expected (ground truth): {dataset[0]['target_codes']}")
print()

simple_result = simple_mapper_func(test_cui)
approx_result = approximate_mapper_func(test_cui)
random_result = random_sampler_func(test_cui)

print("Simple mapper (direct lookup):")
for code in simple_result:
    print(f"  - {code}")

print("\nApproximate mapper (string-based):")
for code in approx_result:
    print(f"  - {code}")

print("\nRandom sampler (baseline):")
for code in random_result:
    print(f"  - {code}")

## Evaluate Vocabulary Mappers

Compare performance of different mappers against real ground truth extracted from RF2.

In [ ]:
# Evaluate simple mapper (direct lookup)
print("=== Evaluating Simple Mapper (Direct Lookup) ===")
results_simple = evaluate_mapper(
    mapper_func=simple_mapper_func,
    dataset=dataset,
    k_values=[1, 3, 5],
)

# Evaluate approximate mapper (string-based matching)
print("\n=== Evaluating Approximate Mapper (String-Based) ===")
results_approx = evaluate_mapper(
    mapper_func=approximate_mapper_func,
    dataset=dataset,
    k_values=[1, 3, 5],
)

# Evaluate random sampler (baseline)
print("\n=== Evaluating Random Sampler (Baseline) ===")
results_random = evaluate_mapper(
    mapper_func=random_sampler_func,
    dataset=dataset,
    k_values=[1, 3, 5],
)

In [ ]:
# Display comprehensive results
print("\n=== Vocabulary Mapping Benchmarking Results ===")

metrics_to_show = [
    ("coverage_rate", "Coverage Rate"),
    ("mrr", "MRR"),
]

print(
    f"\n{'Method':<25} {'Precision@1':<12} {'Recall@1':<12} {'Coverage':<12} {'MRR':<10}"
)
print("-" * 75)

for results, name in [
    (results_simple, "Simple (direct lookup)"),
    (results_approx, "Approximate (string-based)"),
    (results_random, "Random sampler"),
]:
    p1 = results.get("precision@1", 0)
    r1 = results.get("recall@1", 0)
    cov = results.get("coverage_rate", 0)
    mrr = results.get("mrr", 0)
    print(f"{name:<25} {p1:<12.4f} {r1:<12.4f} {cov:<12.4f} {mrr:<10.4f}")

print("\nPrecision and Recall at different K values:")
print(f"{'Method':<25} {'K':<5} {'Precision':<12} {'Recall':<12} {'F1':<10}")
print("-" * 60)

for results, name in [
    (results_simple, "Simple"),
    (results_approx, "Approximate"),
    (results_random, "Random"),
]:
    for k in [1, 3, 5]:
        p = results.get(f"precision@{k}", 0)
        r = results.get(f"recall@{k}", 0)
        f1 = results.get(f"f1@{k}", 0)
        print(f"{name:<25} {k:<5} {p:<12.4f} {r:<12.4f} {f1:<10.4f}")

print(f"\nTotal samples evaluated: {results_simple['num_samples']}")

## Sample-Level Analysis

Detailed analysis of how each mapper performs on individual concepts.

In [ ]:
# Detailed per-sample analysis for first 5 samples
print("\nSample-level coverage analysis:")

for i, sample in enumerate(dataset[:5]):
    snomed_cui = sample["snomed_cui"]
    expected = set(sample["target_codes"])

    simple_pred = set(simple_mapper_func(snomed_cui))
    approx_pred = set(approximate_mapper_func(snomed_cui))
    random_pred = set(random_sampler_func(snomed_cui))

    def calc_metrics(pred_set, expected_set):
        if not expected_set:
            return 0.0, 0.0
        intersection = len(expected_set & pred_set)
        precision = intersection / len(pred_set) if pred_set else 0.0
        recall = intersection / len(expected_set)
        return precision, recall

    simple_p, simple_r = calc_metrics(simple_pred, expected)
    approx_p, approx_r = calc_metrics(approx_pred, expected)
    random_p, random_r = calc_metrics(random_pred, expected)

    print(f"\nSample {i+1} (CUI: {snomed_cui[:8]}...):")
    print(f"  Expected ({len(expected)} codes): {[c[:20] for c in list(expected)[:3]]}")
    print(f"  Simple: P={simple_p:.2f}, R={simple_r:.2f} [{len(simple_pred)} codes]")
    print(f"  Approx: P={approx_p:.2f}, R={approx_r:.2f} [{len(approx_pred)} codes]")
    print(f"  Random: P={random_p:.2f}, R={random_r:.2f} [{len(random_pred)} codes]")

## Loading Pre-generated Datasets

Load datasets with different sizes for benchmarking.

In [ ]:
from snomed_methods.benchmarking.vocabulary import load_mapping_datasets

# Load all pre-generated datasets using real RF2 data
datasets = load_mapping_datasets(snomed_dir=rf2_path)

for name, data in datasets.items():
    num_with_mappings = sum(1 for s in data if s["has_mapping"])
    print(f"{name}: {len(data)} samples ({num_with_mappings} with mappings)")